# Chapter 1 · Hashing, Token Counting, and LLM Context
### From LeetCode to Learning Systems

**Author:** Deb Paul · [github.com/dpaul0501](https://github.com/dpaul0501) · [#MLDSA](https://linkedin.com/in/dpaul0501)

---

**What you'll implement in this notebook:**
1. HashMap from scratch (with chaining + resize)
2. LRU Cache — the production caching primitive
3. Token vocabulary — what tokenizers actually are
4. Feature hashing — the hashing trick
5. Agent tool registry
6. LeetCode patterns: Two Sum, Group Anagrams, Subarray Sum = K
7. Practice problems (try them yourself!)

**ML connections:** Tokenizers · KV Cache · Feature Stores · Agent Tool Dispatch


---
## Section 1 · HashMap from Scratch

Before using Python's `dict`, let's build one. This demystifies every HashMap problem and explains what's happening inside your feature store and tokenizer.

In [ ]:
class HashMap:
    """
    Separate chaining HashMap.
    Implements what Python's dict does under the hood.
    """
    def __init__(self, capacity=16):
        self.capacity = capacity
        self.size = 0
        self.buckets = [[] for _ in range(capacity)]  # list of (k, v) pairs
        self.load_factor_limit = 0.67

    def _hash(self, key):
        return hash(key) % self.capacity

    def put(self, key, value):
        idx = self._hash(key)
        bucket = self.buckets[idx]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket[i] = (key, value)  # update
                return
        bucket.append((key, value))       # insert
        self.size += 1
        if self.size / self.capacity > self.load_factor_limit:
            self._resize()

    def get(self, key, default=None):
        idx = self._hash(key)
        for k, v in self.buckets[idx]:
            if k == key:
                return v
        return default

    def delete(self, key):
        idx = self._hash(key)
        bucket = self.buckets[idx]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket.pop(i)
                self.size -= 1
                return

    def _resize(self):
        old_buckets = self.buckets
        self.capacity *= 2
        self.buckets = [[] for _ in range(self.capacity)]
        self.size = 0
        for bucket in old_buckets:
            for k, v in bucket:
                self.put(k, v)

    def __repr__(self):
        items = [(k, v) for b in self.buckets for k, v in b]
        return f"HashMap({dict(items)})"


# Test it
hm = HashMap()
hm.put("token", 42)
hm.put("model", 99)
hm.put("token", 100)  # update
print(hm.get("token"))   # 100
print(hm.get("model"))   # 99
print(hm.get("missing", -1))  # -1
print(f"Size: {hm.size}, Capacity: {hm.capacity}")

---
## Section 2 · LRU Cache (LeetCode #146)

The LRU Cache is the most important HashMap variant in ML systems:
- **Model serving:** cache recent inference results
- **Tokenizer cache:** cache encoded sequences  
- **KV cache management:** evict least recently used attention states
- **Feature store:** cache hot features

Key insight: `OrderedDict` gives us HashMap + access-order tracking in one structure.

In [ ]:
from collections import OrderedDict

class LRUCache:
    """O(1) get and put using HashMap + doubly linked list."""
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = OrderedDict()

    def get(self, key: int) -> int:
        if key not in self.cache:
            return -1
        self.cache.move_to_end(key)   # mark as recently used
        return self.cache[key]

    def put(self, key: int, value) -> None:
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            self.cache.popitem(last=False)  # evict LRU

    def __repr__(self):
        return f"LRUCache(cap={self.capacity}, items={list(self.cache.keys())})"


# Simulate an embedding cache for a model serving system
embed_cache = LRUCache(capacity=3)
embed_cache.put("hello", [0.1, 0.2, 0.3])
embed_cache.put("world", [0.4, 0.5, 0.6])
embed_cache.put("foo",   [0.7, 0.8, 0.9])
print(embed_cache)

embed_cache.get("hello")              # promote "hello"
embed_cache.put("bar", [1.0, 1.1, 1.2])  # evicts "world" (LRU)
print(embed_cache)

print(embed_cache.get("world"))  # -1 (evicted)

---
## Section 3 · Token Vocabulary — What Tokenizers Actually Are

A tokenizer is two HashMaps:
- `token_to_id`: string → int (for encoding)
- `id_to_token`: int → string (for decoding)

When you call `tokenizer.encode("hello world")`, you're doing HashMap lookups.

In [ ]:
class Vocabulary:
    """Bidirectional HashMap: the core of every NLP pipeline."""
    def __init__(self):
        self.token_to_id = {}
        self.id_to_token = {}
        self._next_id = 0
        for special in ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]:
            self.add(special)

    def add(self, token: str) -> int:
        if token not in self.token_to_id:
            self.token_to_id[token] = self._next_id
            self.id_to_token[self._next_id] = token
            self._next_id += 1
        return self.token_to_id[token]

    def encode(self, text: str) -> list:
        return [
            self.token_to_id.get(t, self.token_to_id["<UNK>"])
            for t in text.split()
        ]

    def decode(self, ids: list) -> str:
        return " ".join(self.id_to_token.get(i, "<UNK>") for i in ids)

    def __len__(self):
        return self._next_id


# Build vocabulary from a tiny corpus
vocab = Vocabulary()
corpus = ["the cat sat on the mat", "the dog sat on the log"]
for sentence in corpus:
    for token in sentence.split():
        vocab.add(token)

print(f"Vocab size: {len(vocab)}")
encoded = vocab.encode("the cat sat")
print(f"Encoded: {encoded}")
print(f"Decoded: {vocab.decode(encoded)}")
print(f"OOV token: {vocab.encode('unseen word')}")

---
## Section 4 · Feature Hashing — The Hashing Trick

In ML, you often have high-cardinality categorical features (user IDs, product SKUs). You can't build a vocabulary in advance.

**Solution:** hash the feature directly to a bucket index. Fixed memory, no vocabulary needed. Used in spam filters, ad click prediction, and online learning.

In [ ]:
import hashlib

class FeatureHasher:
    """Maps arbitrary string features to a fixed-size sparse vector."""
    def __init__(self, n_features: int = 2**10):  # small for demo
        self.n_features = n_features

    def _hash(self, feature: str) -> int:
        h = int(hashlib.md5(feature.encode()).hexdigest(), 16)
        return h % self.n_features

    def transform(self, features: list) -> dict:
        """Returns sparse feature vector as {index: value}"""
        vec = {}
        for f in features:
            idx = self._hash(f)
            vec[idx] = vec.get(idx, 0) + 1.0
        return vec


hasher = FeatureHasher(n_features=1024)

# Encode user features without any vocabulary
user_features = ["country:US", "device:mobile", "age_bucket:25-34", "country:US"]
sparse_vec = hasher.transform(user_features)
print(f"Sparse vector: {sparse_vec}")
print(f"country:US appears at index: {hasher._hash('country:US')}")

# Compare two users by dot product
def sparse_dot(a: dict, b: dict) -> float:
    return sum(a.get(k, 0) * v for k, v in b.items())

user2 = hasher.transform(["country:US", "device:desktop", "age_bucket:25-34"])
print(f"User similarity: {sparse_dot(sparse_vec, user2):.2f}")

---
## Section 5 · Agent Tool Registry

When an LLM agent calls a tool, it dispatches through a registry — a HashMap from tool name to callable. This is the core of every tool-using agent.

In [ ]:
class ToolRegistry:
    """HashMap from tool name → callable + schema."""
    def __init__(self):
        self._tools = {}
        self._schemas = {}

    def register(self, name: str, fn, schema: dict):
        self._tools[name] = fn
        self._schemas[name] = schema
        print(f"  ✓ Registered: {name}")

    def call(self, name: str, args: dict):
        if name not in self._tools:
            available = list(self._tools.keys())
            raise ValueError(f"Unknown tool '{name}'. Available: {available}")
        return self._tools[name](**args)

    def list_tools(self) -> list:
        return list(self._tools.keys())


# Register tools
print("Registering tools...")
registry = ToolRegistry()
registry.register(
    "search_web",
    fn=lambda query: f"[Results for: {query}]",
    schema={"name": "search_web", "params": {"query": "string"}}
)
registry.register(
    "calculate",
    fn=lambda expr: eval(expr),
    schema={"name": "calculate", "params": {"expr": "string"}}
)

# Agent dispatches a tool call
print("\nAgent tool calls:")
print(registry.call("search_web", {"query": "PyData Seattle"}))
print(registry.call("calculate", {"expr": "2 ** 10"}))

# What happens with an unknown tool?
try:
    registry.call("searh_web", {"query": "typo"})  # typo!
except ValueError as e:
    print(f"Error: {e}")

---
## Section 6 · LeetCode Patterns

The four canonical HashMap LeetCode problems. Each one is a real production pattern in disguise.

In [ ]:
# ── Problem 1: Two Sum (LeetCode #1) ──────────────────────────────────────
# Production analog: find matching embedding pairs, lookup by feature complement

def two_sum(nums: list, target: int) -> list:
    seen = {}  # value → index
    for i, n in enumerate(nums):
        complement = target - n
        if complement in seen:
            return [seen[complement], i]
        seen[n] = i
    return []

print("Two Sum:", two_sum([2, 7, 11, 15], 9))   # [0, 1]
print("Two Sum:", two_sum([3, 2, 4], 6))         # [1, 2]

In [ ]:
# ── Problem 2: Group Anagrams (LeetCode #49) ──────────────────────────────
# Production analog: group synonymous features, cluster equivalent representations

from collections import defaultdict

def group_anagrams(strs: list) -> list:
    groups = defaultdict(list)
    for s in strs:
        key = tuple(sorted(s))  # canonical form
        groups[key].append(s)
    return list(groups.values())

print("Group Anagrams:", group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]))

In [ ]:
# ── Problem 3: Subarray Sum Equals K (LeetCode #560) ─────────────────────
# Production analog: find time windows in a metric stream that sum to a threshold

def subarray_sum(nums: list, k: int) -> int:
    count = 0
    prefix = 0
    freq = defaultdict(int)
    freq[0] = 1  # empty prefix

    for n in nums:
        prefix += n
        count += freq[prefix - k]
        freq[prefix] += 1

    return count

print("Subarray Sum = 2:", subarray_sum([1, 1, 1], 2))  # 2
print("Subarray Sum = 3:", subarray_sum([1, 2, 3], 3))  # 2

In [ ]:
# ── Problem 4: Longest Consecutive Sequence (LeetCode #128) ───────────────
# Production analog: find longest uninterrupted run of IDs in a user session log

def longest_consecutive(nums: list) -> int:
    num_set = set(nums)
    best = 0
    for n in num_set:
        if n - 1 not in num_set:  # start of a sequence
            length = 1
            while n + length in num_set:
                length += 1
            best = max(best, length)
    return best

print("Longest Consecutive:", longest_consecutive([100, 4, 200, 1, 3, 2]))  # 4

---
## Section 7 · Practice Problems (Try Yourself)

Implement these in the cells below. Solutions are in the chapter text.

In [ ]:
# Practice 1: Build a bag-of-words vectorizer from scratch
# Input:  list of documents (strings)
# Output: sparse vectors as list of dicts {token: count}
# Bonus:  add TF-IDF weighting

def bag_of_words(documents: list) -> list:
    # YOUR CODE HERE
    pass

docs = ["the cat sat", "the dog ran", "cat dog cat"]
# Expected: [{"the": 1, "cat": 1, "sat": 1}, {"the": 1, "dog": 1, "ran": 1}, {"cat": 2, "dog": 1}]
print(bag_of_words(docs))

In [ ]:
# Practice 2: Fuzzy tool name matcher using edit distance
# When an LLM produces "searh_web" instead of "search_web",
# find the closest valid tool name

def edit_distance(s1: str, s2: str) -> int:
    # YOUR CODE HERE
    pass

def fuzzy_tool_match(query: str, tool_names: list) -> str:
    # YOUR CODE HERE
    pass

tools = ["search_web", "run_python", "read_file", "write_file"]
print(fuzzy_tool_match("searh_web", tools))   # search_web
print(fuzzy_tool_match("run_pythn", tools))   # run_python

In [ ]:
# Practice 3: LRU Cache with TTL (time-to-live)
# Extend LRUCache so entries expire after `ttl` seconds
# Useful for feature stores with staleness constraints

import time

class LRUCacheTTL:
    def __init__(self, capacity: int, ttl: float):
        # YOUR CODE HERE
        pass

    def get(self, key):
        # YOUR CODE HERE  
        pass

    def put(self, key, value):
        # YOUR CODE HERE
        pass

# Test
cache = LRUCacheTTL(capacity=2, ttl=1.0)
cache.put("feature_user_1", {"age": 25, "country": "US"})
print(cache.get("feature_user_1"))  # should return the dict
time.sleep(1.5)
print(cache.get("feature_user_1"))  # should return None (expired)

---
## Summary

| Concept | LeetCode | ML System |
|---------|----------|-----------|
| HashMap | Two Sum, Group Anagrams | Feature store, embedding lookup |
| LRU Cache | LeetCode #146 | KV cache, model serving cache |
| Frequency map | Top K Frequent | Vocabulary building, token counting |
| Bidirectional map | — | Tokenizer encode/decode |
| Hash dispatch | — | Agent tool registry |
| Feature hashing | — | Hashing trick for high-cardinality features |

---

**Next notebook:** [Chapter 2 — Top-K, Vector Search, and RAG Retrieval](./chapter02_topk.ipynb)

**Book:** [dpaul0501.github.io/learning-systems-from-scratch](https://dpaul0501.github.io/learning-systems-from-scratch)  
**LinkedIn:** [#MLDSA](https://linkedin.com/in/dpaul0501) — Day 1 of 60 is live!
